# 167. Two Sum II - Input Array Is Sorted

[Problem](https://leetcode.com/problems/two-sum-ii-input-array-is-sorted/) · difficulty: medium

Eight approaches, all built on the same invariant. This notebook is about the two things the
LeetCode runtimes do not show: **which input shape separates them**, and **why the slowest
submission was never slow at all**.


In [ ]:
import pathlib, sys

# works from the problem directory or the project root
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0167-two-sum-ii-input-array-is-sorted'
sys.path.insert(0, str(ROOT))

from lc.harness import load_solutions

solutions = load_solutions(PROBLEM)
[s.__name__ for s in solutions]


## Summary

| # | Approach | Time | Space | LeetCode |
|---|---|---|---|---|
| 1 | `SolutionHashMap` | O(n) | O(n) | 7 ms |
| 2 | `SolutionTwoPointers` | O(n) | O(1) | 7 ms |
| 3 | `SolutionTwoPointersSkipImpossible` | O(n) | O(1) | 9 ms |
| 4 | `SolutionTwoPointersSkipDuplicates` | O(n) | O(1) | 7 ms |
| 5 | `SolutionBisectJump` | O(d log n) | O(1) | — |
| 6 | `SolutionMidpointProbe` | O(n) | O(1) | 3 ms |
| 7 | `SolutionBisectRunEnd` | O(d log n) | O(1) | 0 ms |
| 8 | `SolutionInlineBisectJump` | O(d log n) | O(1) | 0 ms |

`d` is the number of *distinct* values met.


## One invariant, nine step sizes

Sorted input means a pair that is too small can only be repaired by a larger left value, and a
pair that is too large only by a smaller right one. So each comparison retires one end:

$$numbers_{lo} + numbers_{hi} < target \;\Rightarrow\; \text{no pair using } numbers_{lo} \text{ can reach } target$$

Every approach below 3 differs only in *how many indices it retires per comparison*.


In [ ]:
def trace(numbers, target, limit=8):
    lo, hi = 0, len(numbers) - 1
    steps = 0
    while lo < hi and steps < limit:
        total = numbers[lo] + numbers[hi]
        verdict = 'found' if total == target else ('lo++' if total < target else 'hi--')
        print(f'lo={lo:>2} hi={hi:>2}  {numbers[lo]:>3} + {numbers[hi]:>3} = {total:>4}  {verdict}')
        if total == target:
            return [lo + 1, hi + 1]
        lo, hi = (lo + 1, hi) if total < target else (lo, hi - 1)
        steps += 1
    print('...')

trace([2, 3, 4], 6)


## The constraints decide the algorithm

`numbers[i]` is capped at ±1000 while the array runs to 30 000 entries. So at most **2001
distinct values** exist — a long input is necessarily mostly runs of duplicates, and every
step that crosses a whole run in one move converts an O(n) walk into an O(d log n) one with
`d ≤ 2001`. That is the entire difference between the 7 ms group and the 0 ms one.


In [ ]:
numbers = [-1] * 29998 + [1, 1]
print('length          ', len(numbers))
print('distinct values ', len(set(numbers)))
print('longest run     ', max(len(list(g)) for _, g in __import__('itertools').groupby(numbers)))


## Which shape separates them

Three inputs at the constraint limit: a wall of duplicates with the answer last, two equal runs
meeting in the middle, and the densest all-distinct array the constraints allow.

Approach 2 is capped at 5 s and approach 8 hangs outright, so both run under a timer.


In [ ]:
import signal, time

class Timeout(Exception):
    pass

signal.signal(signal.SIGALRM, lambda *_: (_ for _ in ()).throw(Timeout()))

def timed(solution, numbers, target, budget=5.0):
    signal.setitimer(signal.ITIMER_REAL, budget)
    try:
        start = time.perf_counter()
        solution().twoSum(list(numbers), target)
        return (time.perf_counter() - start) * 1e3
    except Timeout:
        return float('inf')
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)

shapes = {
    'wall of 1s, answer last': ([1] * 29998 + [2, 3], 5),
    'two runs meeting':        ([-1] * 15000 + [1] * 15000, 0),
    'all 2001 distinct':       (list(range(-1000, 1001)), 1999),
}

print(f"{'approach':<36}" + ''.join(f'{name:>26}' for name in shapes))
for solution in solutions:
    row = ''
    for numbers, target in shapes.values():
        ms = timed(solution, numbers, target)
        row += f"{'timeout' if ms == float('inf') else f'{ms:.3f} ms':>26}"
    print(f'{solution.__name__:<36}{row}')


Two readings worth keeping:

- On the wall of duplicates the four jumping approaches finish in ~0.03 ms against ~1–1.8 ms
  for the linear walkers: a 50x gap that the judge's 7 ms vs 3 ms barely hints at.
- On all-distinct input `SolutionBisectRunEnd` — the 0 ms submission — is the **slowest** of the
  working approaches. Every run has length one, so each binary search pays O(log n) to advance a
  single index. `SolutionBisectJump` avoids that because its jump is driven by the target rather
  than by run boundaries, and it wins on both shapes.


## Takeaway

- Read the constraints before optimizing. ±1000 over 30 000 entries *is* the problem: it
  guarantees duplicate runs, which is the only reason run-skipping beats a plain walk.
- A 0 ms submission is a statement about one test distribution. `SolutionBisectRunEnd` wins
  there and loses by 10x on input the judge never generates; `SolutionBisectJump` is the one
  that holds up on both.
